## Generet marker genelists from databases

In [13]:
# eval "$(conda shell.bash hook)"
# conda init
# conda activate /work/islet_cartography_scrna/scrna_cartography_py_analysis
# python -m ipykernel install --user --name scrna_cartography_py_analysis --display-name "py_analysis"

Load library

In [14]:
# Path-related libraries
import os
import glob
from pyhere import here  # For reproducible relative paths
import sys # system specific parameters
from pathlib import Path # File system paths

# Numerical operations
import numpy as np        # For numerical computations and array manipulations
import pandas as pd

# Download from internet
import requests # http

# Custom modules and functions
sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
import my_anndata as ma                    # Custom AnnData utilities
import misc as misc   

Parameters

In [15]:
# create directory for marker genes
ma.create_directories(here('data/marker_database'))
ma.create_directories(here('data/marker_database/harmonized'))
markers_dir = Path(here('data/marker_database'))
harmo_dir = Path(here('data/marker_database/harmonized'))

/work/islet_cartography_scrna/data/marker_database Directory already exists!
/work/islet_cartography_scrna/data/marker_database/harmonized Directory already exists!


Cell rename dictionary

In [17]:
cell_type_map = {
    "acinar": ["acinar", "acinar cell", "acinar_cells"],
    "alpha": ["alpha", "alpha cell", "α cell", "alpha_cells", "alpha_cell_α_cell"],
    "beta": ["beta", "beta cell", "β cell", "beta_cells", "beta_cell_β_cell", "beta_like_cell"],
    "delta": ["delta", "delta cell", "δ cell", "delta_cells", "delta_cell"],
    "gamma": ["gamma", "gamma (pp) cell", "gamma_cell", "pp_cell", "pancreatic_polypeptide_cell", "gamma_pp_cells", "gamma_delta_t_cells"],
    "ductal": ["ductal", "ductal cell", "ductal_cells", "ductal_epithelial_cell"],
    "endothelial": ["endothelial", "endothelial cell", "lymphatic_endothelial_cell"],
    "stellate": ["stellate", "stellate cell", "quiescent_stellate", "activated_stellate", "pancreatic_stellate_cell"],
    "schwann": ["schwann", "schwann cell", "peri-islet_schwann_cell"],
    "epsilon": ["epsilon", "epsilon cell"],
    "immune": ["immune", "immune_cell"],
    "t_lymphocyte": ["t_cells", "t_cell"],
    "t_memory": ["t_memory_cells"],
    "t_helper": ["t_helper_cell"],
    "t_h1": ["t_helper_1_th1_cell"],
    "t_h2": ["t_helper_2_th2_cell"],
    "t_reg": ["regulatory_t_treg_cell"],
    "t_cytotoxic": ["cytotoxic_lymphocyte"],
    "nkt": ["natural_killer_t_nkt_cell"],
    "nk": ["natural_killer_cell", "nk_cells"],
    "b_lymphocyte": ["b_cell", "b_cells"],
    "b_lymphocyte_naive": ["b_cells_naive", "b_lymphocyte_naive"], 
    "plasmacytoid_dendritic": ["plasmacytoid_dendritic_cells"],
    "dendritic": ["dendritic_cells", "dendritic_cell"],
    "macrophage": ["macrophage", "macrophages", "macrophage_cell"],
    "pro_inf_macrophage": ["pro_inflammatory_macrophage"],
    "mast": ["mast_cell", "mast_cells"],
    "neutrophil": ["neutrophils"],
    "lymphocyte": ["lymphocyte"],
    "blood_cell": ["blood_cell"],
    "monocyte": ["monocyte", "monocytes"],
    "progenitor": ["progenitor_cell", "pancreatic_progenitor_cell", "undifferentiated_pancreatic_progenitor_cell",
                   "endocrine_progenitor_cell"],
    "stem_cell": ["stem_cell", "pancreatic_stem_cell"],
    "cancer_stem": ["cancer_stem_cell"],
    "epithelial": ["epithelial_cell", "epithelial_ductal_cell", "epithelial_cells"],
    "fibroblast": ["fibroblast"],
    "mesenchymal": ["mesenchymal_cell"],
    "smooth_muscle": ["smooth_muscle_cell"],
    "stellate": ["pancreatic_stellate_cells"],
    "neuron": ["neuron", "neuroendocrine_cell"],
    "pancreas": ["pancreas_cell"],
    "endo_pancreas": ["islet_cell", "endocrine_cell"],
    "exo_pancreas": ["exocrine_cell"],
    "follicular_b_lymphocyte": ["follicular_b_cell"],
    "myeloid": ["myeloid_cell"],
    "pericytes": ["pericytes"],
    "plasma_cell": ["plasma_cells"],
    "basal_cell": ["basal_cell"],
    "ductal_stem_cell": ["pancreatic_ductal_stem_cell"],
    "activated_stellate": ["activated_stellate"],
    "cycling" : ["cycling"],
    "quiescent_stellate": ["quiescent_stellate"],
    "padc_derived_cell_line": ["pancreatic_ductal_adenocarcinoma_pdac_derived_cell_line"]
}

# Reverse the map
reverse_map = {}
for canonical, aliases in cell_type_map.items():
    for a in aliases:
        reverse_map[misc.to_snake_case(a)] = canonical

# save as csv
rows = []
for old, new in reverse_map.items():
    rows.append({"old_cell_name": old, "new_cell_name": new})
        
df = pd.DataFrame(rows)
df.to_csv(os.path.join(markers_dir, 'cell_type_map.csv'), index=False)

### Download pancreatic cells marker genes lists

Panglaodb and cellmarker

In [5]:
# # Make directory
# data_dir = markers_dir

# # Define files and URLs (acess: 2025-10-14)(year/month/day)
# files = {
#     "panglaodb_markers.tsv.gz": "https://panglaodb.se/markers/PanglaoDB_markers_27_Mar_2020.tsv.gz",
#     "cellmarker20.xlsx": "http://www.bio-bigdata.center/CellMarker_download_files/file/Cell_marker_Human.xlsx"
# }

# for filename, url in files.items():
#     filepath = data_dir / filename
#     if not filepath.exists():
#         print(f"Downloading {filename}...")
#         r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, stream=True)
#         r.raise_for_status()
#         with open(filepath, "wb") as f:
#             for chunk in r.iter_content(chunk_size=8192):
#                 f.write(chunk)
#     else:
#         print(f"{filename} already exists, skipping download.")


Azimuth

In [18]:
# Azimuth marker genes https://azimuth.hubmapconsortium.org/references/#Human%20-%20Pancreas (acess: 2025-10-14) 
# manually written as there is no download
azimuth_genes = {
    "cycling": ["UBE2C","TOP2A","CDK1","BIRC5","PBK","CDKN3","MKI67","CDC20","CCNB2","CDCA3"],
    "immune": ["ACP5","APOE","HLA-DRA","TYROBP","LAPTM5","SDS","FCER1G","C1QC","C1QB","SRGN"],
    "quiescent_stellate": ["RGS5","C11orf96","FABP4","CSRP2","IL24","ADIRF","NDUFA4L2","GPX3","IGFBP4","ESAM"],
    "endothelial": ["PLVAP","RGCC","ENG","PECAM1","ESM1","SERPINE1","CLDN5","STC1","MMP1","GNG11"],
    "schwann": ["NGFR","CDH19","UCN2","SOX10","S100A1","PLP1","TSPAN11","WNT16","SOX2","TFAP2A"],
    "activated_stellate": ["COL1A1","COL1A2","COL6A3","COL3A1","TIMP3","TIMP1","CTHRC1","SFRP2","BGN","LUM"],
    "epsilon": ["BHMT","VSTM2L","PHGR1","TM4SF5","ANXA13","ASGR1","DEFB1","GHRL","COL22A1","OLFML3"],
    "gamma": ["PPY","AQP3","MEIS2","ID2","GPC5-AS1","CARTPT","PRSS23","ETV1","PPY2","TUBB2A"],
    "delta": ["SST","RBP4","SERPINA1","RGS2","PCSK1","SEC11C","HHEX","LEPR","MDK","LY6H"],
    "ductal": ["SPP1","MMP7","IGFBP7","KRT7","ANXA4","SERPINA1","LCN2","CFTR","KRT19","SERPING1"],
    "acinar": ["REG1A","PRSS1","CTRB2","CTRB1","REG1B","CELA3A","PRSS2","REG3A","CPA1","CLPS"],
    "beta": ["IAPP","INS","DLK1","INS-IGF2","G6PC2","HADH","ADCYAP1","GSN","NPTX2","C12orf75"],
    "alpha": ["GCG","TTR","PPP1R1A","CRYBA2","TM4SF4","MAFB","GC","GPX3","PCSK2","PEMT"]
}

TF database

Description from website:

> Here, we developed the TF-Marker database (TF-Marker, http://bio.liclab.net/TF-Marker/) which is committed to a comprehensive manual curation of TFs and related markers with experimental evidence in specific cell and tissue types in human. Currently, through reviewing 2,091 published literature, we have manually classified TFs and related markers into five types according to their functions: 1) TF: TFs, which regulate the expression of markers; 2) T Marker: markers, which are regulated by TFs (TF and T Marker pairs can identify cell types more specifically); 3) I Marker: markers, which influence the activity of TFs (I Markers can also influence the development of specific cells and tissues); 4) TFMarker: TFs, which play roles as markers (TFMarkers are cell/tissue-specific TFs used as cell markers in biology experiments); and 5) TF Pmarker: TFs, which play roles as potential markers. By curating thousands of published literature, 5,905 entries including 1,316 TFs, 1,092 T Markers, 473 I Markers, 1,600 TFMarkers and 1,424 TF Pmarkers, were annotated in 383 cell types and 95 tissue types in human. Moreover, TF-Marker divided markers into disease markers and tissue/cell-specific markers. TF-Marker is an elaborate database, which provides TFs and related markers supported by experimental evidence. We believe TF-Marker will provide strong support for research into cell/tissue-specific TFs and related markers.


Description of the types of markers in this database:

- **Type 1 markers (TF):** TFs which regulate expression of the markers
- **Type 2 (T marker):** Markers which are regulated by the TF
- **Type 3 (I marker):** marker which influence the activity of the TF
- **Type 5 (TF Pmarker):** TFs which play roles as potential markers
- **Type 4 (TFMarker):** TFs which play roles as markers

Download files (access 2025-11-17) (year/month/day)

These files were manually downloaded, because the server blocks direct automated download requests without a valid session and referer.

URLS:

- "data/marker_database/all_tf_markers.txt": "http://www.licpathway.net/TF-Marker/public/download/main_download.csv"
- "data/marker_database/tf_i_markers.txt": "http://www.licpathway.net/TF-Marker/public/download/download_I%20Marker.csv"
- "data/marker_database/tf_t_markers.txt": "http://www.licpathway.net/TF-Marker/public/download/download_T%20Marker.csv"
- "data/marker_database/tf_tf.txt": "http://www.licpathway.net/TF-Marker/public/download/download_TF.csv"
- "data/marker_database/tf_tf_pmarker.txt": "http://www.licpathway.net/TF-Marker/public/download/download_TF%20Pmarker.csv"
- "data/marker_database/tf_tf_marker": "http://www.licpathway.net/TF-Marker/public/download/download_TFMarker.csv"

### Load data

In [19]:
# Gene lists
panglao_df = pd.read_csv(here('data/marker_database/panglaodb_markers.tsv.gz'), sep='\t', compression='gzip')
tf_paths = glob.glob(os.path.join(markers_dir, "tf_*.txt"))

tf_df = {}
for file in tf_paths:
    name = Path(file).stem
    tf_df[name] = pd.read_csv(file, sep=",", dtype=str)
    
cellmarker_df = pd.read_excel(here('data/marker_database/cellmarker20.xlsx'), sheet_name='human')

# renameing map
reverse_map = pd.read_csv(os.path.join(markers_dir, 'cell_type_map.csv'))
reverse_map = reverse_map.set_index('old_cell_name')['new_cell_name'].to_dict()

### Make dictionary for dataframes

Panglao

In [20]:
panglao_df_flt = panglao_df[
    (panglao_df["organ"].isin(['Pancreas', 'Immune system', 'Vasculature'])) &
    (panglao_df['species'].isin(['Hs', 'Mm Hs'])) &
    (panglao_df['sensitivity_human'] > 0.5) &
    (panglao_df['specificity_human'] < 0.1)
].copy()

# Standardize cell type names using reverse_map
panglao_df_flt.rename(columns={'cell type': 'old_cell_name'}, inplace=True)
panglao_df_flt['cell_type_simple'] = panglao_df_flt['old_cell_name'].apply(
    lambda x: reverse_map.get(misc.to_snake_case(x), misc.to_snake_case(x))
)

# Make dictionary
panglao_genes = panglao_df_flt.groupby('cell_type_simple')['official gene symbol'].apply(list).to_dict()

Cell Marker

In [21]:
cellmarker_df_flt = cellmarker_df[
    (cellmarker_df['tissue_type'].isin(['Pancreas', 'Pancreatic acinar tissue', 'Pancreatic duct'])) &
    (cellmarker_df['cancer_type'] == 'Normal') &
    (cellmarker_df['cell_type'] == 'Normal cell')].copy()

# Standardize cell type names using reverse_map
cellmarker_df_flt.rename(columns={'cell_name': 'old_cell_name'}, inplace=True)
cellmarker_df_flt['cell_type_simple'] = cellmarker_df_flt['old_cell_name'].apply(
    lambda x: reverse_map.get(misc.to_snake_case(x), misc.to_snake_case(x))
)

cellmarker_df_flt = (cellmarker_df_flt.dropna(subset=['cell_type_simple', 'Symbol', 'cellontology_id'])
                     .reset_index()[['cell_type_simple', 'marker', 'cellontology_id']]
                     .drop_duplicates())

cellmarker_df_flt.rename(columns={'cellontology_id': 'cell_ontology_id',
                                 'cell_type_simple': 'cell_type',
                                 'marker': 'gene'}, inplace=True)

TF df database

In [22]:
tf_df_pancreas = {}

for name, df in tf_df.items():
    df_flt = df.copy()[(df["Tissue Type"].isin(["Pancreas"]) &
                        df["Cell Type"].isin(["Normal cell"]))] 
    df_flt .rename(columns={'Cell Name': 'old_cell_name'}, inplace=True)
    df_flt['cell_type_simple'] = df_flt ['old_cell_name'].apply(
        lambda x: reverse_map.get(misc.to_snake_case(x), misc.to_snake_case(x))
    )

    df_flt = (df_flt.dropna(subset=['cell_type_simple', 'Gene Name', 'CellOntologyID'])
                     .reset_index()[['cell_type_simple', 'Gene Name', 'CellOntologyID']]
                     .drop_duplicates())

    df_flt.rename(columns={'CellOntologyID': 'cell_ontology_id',
                          'cell_type_simple': 'cell_type',
                          'Gene Name': 'gene'}, inplace=True)
    
    tf_df_pancreas[name] = df_flt

Save genelists

In [23]:
# Azimuth
rows = []
for celltype, genes in azimuth_genes.items():
    for gene in genes:
        rows.append({"cell_type": celltype, "gene": gene})

df = pd.DataFrame(rows)
df.to_csv(os.path.join(harmo_dir, 'azimuth_genes.csv'), index=False)

# Panglao
rows = []
for celltype, genes in panglao_genes.items():
    for gene in genes:
        rows.append({"cell_type": celltype, "gene": gene})

df = pd.DataFrame(rows)
df.to_csv(os.path.join(harmo_dir, 'panglao_genes.csv'), index=False)

# Cellmarker
cellmarker_df_flt.to_csv(os.path.join(harmo_dir, 'cellmarker_genes.csv'), index=False)

# TF database
for name, df in tf_df_pancreas.items():
    df.to_csv(os.path.join(harmo_dir, f'{name}_genes.csv'), index=False)

How to load genelists again

In [24]:
# path to genelists
genelists_path = glob.glob(os.path.join(harmo_dir, '*.csv'))

# Load and convert to dictionary
genelists = {}
for file in genelists_path:
    name = Path(file).stem.replace('_genes', '')
    df = pd.read_csv(file, sep=",", dtype=str)
    genelists[name] = df.groupby('cell_type')['gene'].apply(list).to_dict()